# Successive Cancellation List

In [1]:
import itertools
import logging
import random
from concurrent.futures import ProcessPoolExecutor
from wrappers.polar_wrapper import (
    polar_code_p2, get_logical_error_on_accepted_states, divide_half_list,
    get_q1prep_accepted_states, get_logical_error_on_accepted_states_SCL
)
from wrappers.stim_wrapper import (
    simulate_stim_polar_code_normal,
    
    calculate_logical_error_result_polar_normal,
    simulate_batch_and_save_result_polar_normal,
    generate_qiskit_polar_code, compiled_to_qiskit_hardware,
    find_and_delete_files
)

import pandas as pd
import os
import glob
import sys

from qiskit_ibm_runtime import QiskitRuntimeService

import collections
import numpy as np

In [2]:
# Import your compiled Cython wrappers
from Encoders.polar import PyEncoderPolar
# Adjust the import below if your Decoder wrapper class is named differently
from Decoders.SCL import PyDecoderPolarSCL 

In [3]:
n = 3
lstate = "z"
sim_type = "normal"
i = 7
p_error = 0
shots = 1e4
seed = 12345

zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
zpos_list[n] = i-1

results = simulate_stim_polar_code_normal(n, lstate, sim_type, i, p_error, shots, seed)

# flipped_results = {bit_str[::-1]: count for bit_str, count in results.items()}
sum(results.values())

10000

In [4]:
zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
zpos_list[n] = i-1

print(get_logical_error_on_accepted_states(n, lstate.upper(), results, zpos_list))

accepted_states, data_qubit_states = get_q1prep_accepted_states(n, lstate, results, zpos_list)
sum(data_qubit_states.values())

(10000, 0, 0, 1.0, 0, 0.03727079299278557)


10000

In [5]:
print(accepted_states)
print(data_qubit_states)

{'00110000001111001110': 42, '11111100001100111111': 31, '11111111000000000111': 48, '11001100000011110101': 38, '11110011110000111110': 41, '11001100000011111111': 36, '00111100111111111001': 40, '11111100001100110001': 40, '00110011000011111000': 32, '00001100110000111010': 49, '11000000110011000110': 30, '11000000110011001111': 49, '00001111111100001100': 53, '00110011000011110110': 28, '00111111110011000000': 38, '11110000111100000011': 44, '11110000111100000100': 43, '00001111111100000100': 40, '11111111000000000000': 52, '00000000000000001101': 43, '11001111001111000011': 33, '00111111110011001011': 34, '11111100001100110110': 44, '00001100110000110110': 39, '00110011000011110001': 42, '11110011110000110001': 50, '00000000000000001001': 29, '11111100001100111101': 32, '11110011110000111011': 48, '00110000001111000000': 41, '11111100001100110100': 41, '11111111000000000011': 35, '00000000000000000010': 47, '11000000110011001011': 48, '11110000111100000110': 50, '110011000000111110

In [6]:
import numpy as np

def calculate_logical_error_rate(data_qubit_states, decoder, p_error, K, expected_info_bits=None):
    """
    Decodes the accepted Stim measurements and calculates the Logical Error Rate (LER).
    
    Args:
        data_qubit_states (dict): e.g., {'00111001': 2, ...}
        decoder: The initialized PyDecoderPolarSCL object.
        p_error (float): Physical error rate used for LLR calculation.
        K (int): Number of information bits.
        expected_info_bits (list/array): The expected K bits (default is all 0s for |0>_L).
        
    Returns:
        total_trials (int): Total number of accepted measurements.
        logical_errors (int): Number of measurements that decoded incorrectly.
        ler (float): Logical Error Rate.
    """
    # By default, state preparation of |0> or |+> usually expects the K info bits to be 0
    if expected_info_bits is None:
        expected_info_bits = np.zeros(K, dtype=np.int32)
    else:
        expected_info_bits = np.array(expected_info_bits, dtype=np.int32)

    # Calculate LLR magnitude from the physical error rate
    if p_error == 0:
        llr_mag = 10.0 # High confidence for noiseless
    else:
        llr_mag = np.log((1 - p_error) / p_error)

    total_trials = 0
    logical_errors = 0

    print(f"{'Measurement':<12} | {'Count':<5} | {'Decoded V_K':<11} | {'Status'}")
    print("-" * 45)

    for bit_str, count in data_qubit_states.items():

        
        # 1. Convert string to NumPy array of integers
        stim_measurements = np.array([int(b) for b in bit_str])

        # 2. Map hard bits to soft LLRs
        # 0 -> +llr_mag, 1 -> -llr_mag
        Y_N_LLRs = np.where(stim_measurements == 0, llr_mag, -llr_mag).astype(np.float64)

        # 3. Decode
        V_K_hat = np.zeros(K, dtype=np.int32)
        decoder.decode(Y_N_LLRs, V_K_hat, 0) # frame_id = 0

        # 4. Check if the decoder successfully recovered the intended state
        if np.array_equal(V_K_hat, expected_info_bits):
            status = "✅ Pass"
        else:
            status = "❌ FAIL"
            logical_errors += count # Add the frequency of this error!
            # Optional: Print the first few results just to see what's happening
            print(f"{bit_str:<12} | {count:<5} | {str(V_K_hat):<11} | {status}")


        total_trials += count
        
        
    # Calculate final LER
    ler = logical_errors / total_trials if total_trials > 0 else 0.0

    print("-" * 45)
    print(f"Total Accepted Trials : {total_trials}")
    print(f"Logical Errors        : {logical_errors}")
    print(f"Logical Error Rate    : {ler:.6f}")

    return total_trials, logical_errors, ler

In [19]:
# --- Your Setup ---
n = 5
lstate = "z"
sim_type = "m1"
i = 2
# i = 7
p_error = 0.001
shots = 1e1
seed = 1234
K = 1
L = 4

def call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L):

    N = 2**n  

    zpos_list = [-1, -1, 1, 3, 6, 7, 22, 15, 90, 31, 362]
    zpos_list[n] = i-1

    if lstate.upper() == "Z":
        zpos = zpos_list[n]
    elif lstate.upper() == "X":
        zpos = zpos_list[n]
    
    print(zpos)

    # 2. Create a mask of ALL Trues (all frozen)
    frozen_bits_mask = np.ones(N, dtype=np.int32)

    # 3. Punch a hole EXACTLY at zpos for the info bit
    frozen_bits_mask[zpos] = 0
    # frozen_bits_mask[zpos:] = 0
    frozen_bits_mask = list(frozen_bits_mask)
    # frozen_bits_mask[N-1-zpos] = 1

    print(frozen_bits_mask)

    results = simulate_stim_polar_code_normal(n, lstate, sim_type, i, p_error, shots, seed)
    accepted_states, data_qubit_states = get_q1prep_accepted_states(n, lstate, results, zpos_list)

    # Initialize the decoder
    decoder = PyDecoderPolarSCL(K, N, L, frozen_bits_mask)

    count_accept, count_logerror, count_undecided, ler, _, _ = get_logical_error_on_accepted_states(n, lstate.upper(), results, zpos_list)
    count_accept_m1, count_logerror_m1, count_undecided_m1, ler_m1, _, _ = get_logical_error_on_accepted_states_SCL(n, lstate.upper(), results, decoder, p_error, zpos_list)

    print(count_accept, count_logerror, count_undecided, 1-ler)
    print(count_accept_m1, count_logerror_m1, count_undecided_m1, 1-ler_m1)

# # --- Run the function ---
# print("\n=== STARTING DECODING AND LER CALCULATION ===")
# total, errors, ler = calculate_logical_error_rate(
#     data_qubit_states=data_qubit_states, 
#     decoder=decoder, 
#     p_error=p_error, 
#     K=K
# )

In [25]:
n = 3
lstate = "x"
# i = 2
i = 5
p_error = 0
shots = 1000
seed = 1000
# seed = random.randint(1, 99999999)
K = 1
L = 1

sim_type = "normal"
print("Normal")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

# sim_type = "m1"
# print("M1")
# call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

Normal
4
[1, 1, 1, 1, 0, 1, 1, 1]
1000 0 0 0.0
1000 0 0 0.0


frozen_bits: 1 1 1 1 0 1 1 1 


In [9]:
AAAaa

NameError: name 'AAAaa' is not defined

In [ ]:
n = 3
lstate = "z"
# i = 2
i = 7
p_error = 1e-2
shots = 1e4
# seed = 10000
seed = random.randint(1, 99999999)
seed = 2345
K = 1
L = 4

sim_type = "normal"
print("Normal")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L, 0)

# sim_type = "m1"
# print("M1")
# call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

Normal


frozen_bits: 0 1 1 1 1 1 1 1 


6823 7 14 0.0010259416678880529
6823 3394 0 0.4974351458302799


In [ ]:
i = 3
print(f"i = {i}, L = {L}")

sim_type = "normal"
print("Normal")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

sim_type = "m1"
print("M1")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

i = 3, L = 4
Normal


frozen_bits: 1 0 0 0 0 0 0 0 


6014 368 702 0.061190555370801425
6014 33 0 0.005487196541403394
M1


frozen_bits: 1 0 0 0 0 0 0 0 


6752 416 805 0.06161137440758291
6752 35 0 0.0051836492890995345


In [ ]:
i = 3
L = 2
print(f"i = {i}, L = {L}")

sim_type = "normal"
print("Normal")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

sim_type = "m1"
print("M1")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

i = 3, L = 2
Normal
(49249, 58, 116, 0.998822311112916, 0, 10.202795016986784)
(49249, 59, 0, 0.9988020061321042, 0, 10.334148295980413)
M1
(49416, 52, 105, 0.9989477092439696, 0, 10.352916577016003)
(49416, 55, 0, 0.9988870001618909, 0, 10.684889324998949)


In [ ]:
i = 3
L = 6
print(f"i = {i}, L = {L}")

sim_type = "normal"
print("Normal")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

sim_type = "m1"
print("M1")
call_experiment(n, lstate, sim_type, i, p_error, shots, seed, K, L)

i = 3, L = 6
Normal
(49249, 58, 116, 0.998822311112916, 0, 11.777862769027706)
(49249, 59, 0, 0.9988020061321042, 0, 12.38726588897407)
M1
(49416, 52, 105, 0.9989477092439696, 0, 10.054339803988114)
(49416, 55, 0, 0.9988870001618909, 0, 10.206581166014075)


In [ ]:
K = 1
N = 2**n
L = 4
p_error = 0.00001
frozen_bits_mask = [True] * (N - K) + [False] * K
decoder = PyDecoderPolarSCL(K, N, L, frozen_bits_mask)


if p_error == 0:
    llr_mag = 10.0 # Arbitrary high confidence for noiseless simulation
else:
    llr_mag = np.log((1 - p_error) / p_error)

# Let's say this is the array of measurements you got from Stim
stim_measurements = np.array([1,1,1,1,1,1,1,1])

# 3. Map the 0s and 1s to +LLR and -LLR
# If bit == 0, it becomes +llr_mag. If bit == 1, it becomes -llr_mag.
Y_N_LLRs = np.where(stim_measurements == 0, llr_mag, -llr_mag).astype(np.float64)

print(f"Hard bits from Stim: {stim_measurements}")
print(f"Soft LLRs for SCL  : {np.round(Y_N_LLRs, 2)}")


# V_K_hat: Array to hold the decoded information bits
V_K_hat = np.zeros(K, dtype=np.int32)


# Pass the LLRs (Y_N) and the output array (V_K_hat) to the C++ decoder
frame_id = 0
decoder.decode(Y_N_LLRs, V_K_hat, frame_id)
V_K_hat


Hard bits from Stim: [1 1 1 1 1 1 1 1]
Soft LLRs for SCL  : [-11.51 -11.51 -11.51 -11.51 -11.51 -11.51 -11.51 -11.51]


array([1], dtype=int32)